# Fundamentals of Machine Learning — Exercise 8
## Dimensionality reduction and latent spaces: PCA, SVD, t-SNE, UMAP

The theory was in the presentation. Today you **try, guess, get surprised and decide** – like a data scientist, not like a student clicking through cells.

### Three kinds of tasks
| | What to do |
|---|---|
| 🔮 **Predict** | Write your guess **before** you run the next cell. We keep a scoreboard at the end. |
| 🛠️ **Do** | A short `TODO` (1–3 lines) followed by a ✅/❌ self-check. |
| 🕵️ **Think** | A trap, a puzzle or a decision. Answer in your own words and **compare with your neighbour**. |



The **⭐** challenge at the end is optional (for the fast ones).

### How to work
1. Complete the `TODO`s; variables set to `None` are waiting for you. Run the **check** cell below.
2. Stuck for more than 5 minutes? Open the 💡 hint. S
3. Run the notebook **top to bottom** (cells depend on each other).

## 0. Setup

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer, load_digits, load_wine
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE, trustworthiness
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif

plt.rcParams["axes.grid"] = True
RANDOM_STATE = 42

# UMAP is not part of scikit-learn. In Colab run first:  !pip install umap-learn
try:
    import umap
    HAS_UMAP = True
except ImportError:
    HAS_UMAP = False
    print("umap-learn is not installed -> the UMAP cells will be skipped (Colab: !pip install umap-learn).")


def check(test, ok="Correct.", hint="Not yet."):
    """Self-check. `test` is a function returning True/False (an error counts as a failure)."""
    try:
        passed = bool(test())
    except Exception as e:
        passed = False
        hint = hint + f"  [{type(e).__name__}: {e}]"
    print(("✅ " + ok) if passed else ("❌ " + hint))

## 1. Theory in a nutshell 

**Why reduce the dimension?** To *see* the data (2D instead of 30D), to remove redundant (correlated) and noisy features, to get fewer inputs for a model, and to compress.

**PCA** finds new axes (principal components): orthogonal directions with the largest variance. The first keeps the most variance, the second the most of what is left, and so on; we keep the first *k*. PCA is **linear**, **unsupervised** (it does not see the labels) and it **learns parameters from data** (mean, scaling, axes) – so it must be fitted on the **training data only**. That is why it lives in a **pipeline**. Technically, PCA is computed by the **SVD** (singular value decomposition) of the centred data: the right singular vectors are the principal axes and σ² / (n − 1) are the variances.

The few numbers PCA gives for every sample are its **latent representation** (a compact "hidden" description). *Encoding* = `transform`, *decoding* = `inverse_transform`.

**t-SNE and UMAP** are non-linear methods for **visualization**: they keep neighbours together in a 2D picture. **Clusters are meaningful; distances between clusters, their sizes and the axes are not.** They depend on parameters (`perplexity`; `n_neighbors`, `min_dist`), and t-SNE cannot embed new points.

| | PCA | t-SNE | UMAP |
|---|---|---|---|
| main job | model input, compression, a quick look | visualization | visualization |
| new data (`transform`) | yes | no | yes |
| parameters | `n_components` | `perplexity` | `n_neighbors`, `min_dist` |

## 2. Look at the data 

Always start here. **Breast cancer**: 569 tumours, 30 numeric measurements of cell nuclei (radius, area, texture, …), label = benign / malignant.

In [ ]:
bc = load_breast_cancer()
X, y = bc.data, bc.target
X_df = pd.DataFrame(X, columns=bc.feature_names)
print("samples, features:", X.shape, "| classes:", dict(zip(bc.target_names, np.bincount(y))))
display(X_df.describe().T[["mean", "std", "min", "max"]].round(2).head(10))

corr = X_df.corr()
plt.figure(figsize=(6.5, 5.5)); plt.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1); plt.colorbar(label="correlation"); plt.grid(False)
plt.title("correlations of the 30 features (radius, perimeter, area...)"); plt.show()

### 🔮 Predict
Before we compute anything: the 30 features are strongly correlated (look at the heatmap). **How many principal components will keep 95 % of the variance?** Write a number between 1 and 30 into `guess_k`.

In [ ]:
guess_k = None   # <- your guess (a whole number from 1 to 30)

**Question 1 🕵️.** Look at the table (`mean`, `std`) and the heatmap. (a) Do we need to **standardize** before PCA? Why? (b) Is PCA a **promising** method for these data?

## 3. PCA step by step 
The recipe from the presentation: **split → scaler and PCA fitted on train only → choose k → pipeline with a model**.

### 🛠️ Task 3.1 – split and scaling
Split the data 70 / 30 (stratified, `random_state=RANDOM_STATE`) into `X_tr, X_te, y_tr, y_te`. Fit a `StandardScaler` **on the training data only** and create `X_tr_s`, `X_te_s`.

<details><summary>💡 Hint</summary>

`train_test_split(X, y, test_size=0.3, random_state=RANDOM_STATE, stratify=y)`. Then `scaler = StandardScaler().fit(X_tr)`; `X_tr_s = scaler.transform(X_tr)`; `X_te_s = scaler.transform(X_te)`. Never `fit` on `X_te`.

</details>

In [ ]:
# TODO
X_tr, X_te, y_tr, y_te = None, None, None, None
scaler = None
X_tr_s, X_te_s = None, None

In [ ]:
check(lambda: X_tr.shape == (398, 30) and X_te.shape == (171, 30), "Split is correct (398 / 171).", "Expected 398 training and 171 test samples.")
check(lambda: np.allclose(X_tr_s.mean(axis=0), 0, atol=1e-9), "Train data are centred.", "Train mean should be 0 after scaling.")
check(lambda: not np.allclose(X_te_s.mean(axis=0), 0, atol=1e-9), "The TEST data are NOT exactly centred: they use the train mean – that is how it should be.",
      "If the test mean is exactly 0, you fitted the scaler on the test data (data leakage).")

### 🛠️ Task 3.2 – how many components?
Fit `PCA()` (all components) on `X_tr_s`. Store the cumulative explained variance in `cum` and the smallest number of components that keep **at least 95 %** in `k95`.

<details><summary>💡 Hint</summary>

`pca_full = PCA().fit(X_tr_s)`; `cum = np.cumsum(pca_full.explained_variance_ratio_)`; `k95 = int(np.argmax(cum >= 0.95) + 1)` (index 0 means 1 component).

</details>

In [ ]:
# TODO
pca_full = None
cum = None
k95 = None

In [ ]:
check(lambda: len(cum) == 30 and abs(cum[-1] - 1) < 1e-9, "Cumulative variance is correct.", "cum should have 30 values ending at 1.")
check(lambda: 9 <= k95 <= 11, f"k95 = {k95}", "Expected about 10.")
if guess_k is None:
    print("🔮 You did not write a guess above - go back and write one next time!")
else:
    print(f"🔮 Your guess: {guess_k}   |   actual: {k95}   |   {'close!' if abs(guess_k - k95) <= 3 else 'far off - why?'}")

fig, ax = plt.subplots(1, 2, figsize=(11, 3.8))
ax[0].bar(range(1, 16), pca_full.explained_variance_ratio_[:15]); ax[0].set_title("scree plot (first 15)"); ax[0].set_xlabel("component")
ax[1].plot(range(1, 31), cum); ax[1].axhline(0.95, color="r", ls="--"); ax[1].set_title("cumulative explained variance"); ax[1].set_xlabel("number of components")
plt.tight_layout(); plt.show()

**Question 2 🕵️.** (a) What is one column of the PCA output – is it one of the original measurements? (b) PCA does not use the labels at all. So why must it still be fitted on the training data only?

### 🛠️ Task 3.3 – the pipeline
Doing the steps by hand is error-prone. Build the **pipeline** `StandardScaler → PCA(k95) → LogisticRegression(max_iter=5000)` as `pipe_pca`, and the same **without** the PCA as `pipe_plain`. Fit both on the **unscaled** `(X_tr, y_tr)` (the pipeline scales itself) and store the test accuracies in `acc_pca`, `acc_plain`.

<details><summary>💡 Hint</summary>

`pipe_pca = make_pipeline(StandardScaler(), PCA(n_components=k95), LogisticRegression(max_iter=5000))`; `pipe_pca.fit(X_tr, y_tr)`; `acc_pca = pipe_pca.score(X_te, y_te)`.

</details>

In [ ]:
# TODO
pipe_pca = None
pipe_plain = None
acc_pca, acc_plain = None, None

In [ ]:
check(lambda: acc_pca > 0.9 and acc_plain > 0.9, "Both pipelines work.", "Both accuracies should be above 0.9.")
print(f"with PCA ({k95} components): {acc_pca:.3f}    without PCA (30 features): {acc_plain:.3f}")
print(f"difference in test samples: {abs(acc_pca - acc_plain) * len(y_te):.0f} of {len(y_te)}")

### 🛠️ Task 3.4 – PCA is SVD
`sklearn` computes PCA through the **SVD** of the centred data: `X_c = U · diag(S) · Vᵀ`. Compute the SVD of the scaled training data `X_tr_s` (use `np.linalg.svd(..., full_matrices=False)`) and store the variances along the axes, `S² / (n − 1)`, in `var_svd`. Then check that they equal the PCA variances.

<details><summary>💡 Hint</summary>

`U, S, Vt = np.linalg.svd(X_tr_s, full_matrices=False)`; `var_svd = S**2 / (X_tr_s.shape[0] - 1)`.

</details>

In [ ]:
# TODO
U, S, Vt = None, None, None
var_svd = None

In [ ]:
check(lambda: np.allclose(var_svd, pca_full.explained_variance_), "σ² / (n−1) = the variances of the principal components.", "var_svd = S**2 / (n - 1)")
check(lambda: np.allclose(np.abs(Vt[0]), np.abs(pca_full.components_[0])), "The first row of Vᵀ is the first principal axis (up to the sign).", "Rows of Vt are the principal axes.")

PCA is the SVD of centred data. In practice: `PCA` is the right tool for dense data; `TruncatedSVD` does the same **without centring**, so it works on sparse matrices (e.g. text) where centring would destroy the sparsity.

## 4. 🕵️ The trap: 82 % accuracy on random labels 

A colleague shows you this experiment. **100 "patients", 2 000 measurements – all of them pure random numbers – and the labels are coin flips.** No model can honestly do better than guessing.

In [ ]:
rng = np.random.RandomState(0)
X_noise = rng.randn(100, 2000)        # 100 patients x 2000 random measurements
y_noise = rng.randint(0, 2, 100)      # labels = coin flips
cv_small = StratifiedKFold(5, shuffle=True, random_state=0)

### 🔮 Predict
What accuracy can any **honest** model reach on these data? Write it into `guess_random` (a number between 0 and 1).

In [ ]:
guess_random = None   # <- your guess

Now the colleague's code: "I first choose the 20 most useful features (the ones that correlate best with the labels), then I evaluate a logistic regression with cross-validation."

In [ ]:
X_sel = SelectKBest(f_classif, k=20).fit_transform(X_noise, y_noise)      # choose 20 'best' features using ALL data and labels
leaky_acc = cross_val_score(LogisticRegression(max_iter=2000), X_sel, y_noise, cv=cv_small).mean()
print(f"colleague's accuracy: {leaky_acc:.2f}   (your guess for an honest model: {guess_random})")

**Question 3 🕵️.** The accuracy is far above your guess, on random data! What did the colleague do wrong, and what does it have to do with the scaler and PCA? (Discuss with your neighbour before you write.)

✍️ **Your answer:** *(double-click this cell and write here)*

### 🛠️ Task 4.1 – fix it with a pipeline
Put the selection **inside** a pipeline `make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=2000))` and evaluate it with `cross_val_score(..., X_noise, y_noise, cv=cv_small)`. The pipeline repeats the selection on the training part of every fold. Store the mean in `honest_acc`.

<details><summary>💡 Hint</summary>

`pipe = make_pipeline(SelectKBest(f_classif, k=20), LogisticRegression(max_iter=2000))`; `honest_acc = cross_val_score(pipe, X_noise, y_noise, cv=cv_small).mean()` – note that the input is the **full** `X_noise` with 2000 columns.

</details>

In [ ]:
# TODO
honest_acc = None

In [ ]:
check(lambda: honest_acc < 0.65, f"Honest accuracy = {honest_acc:.2f}: around guessing, as it should be.", "Expected about 0.5. Use the pipeline and the FULL X_noise.")
print(f"leaky: {leaky_acc:.2f}   honest: {honest_acc:.2f}")

## 5. Does PCA help? Predict, test, decide 

Back to the breast cancer data. We compare three models **with and without PCA** by cross-validation (the pipeline refits the scaler and the PCA in every fold).

### 🔮 Predict
For each model: will the accuracy with **10 components (a third of the features)** be `"better"`, `"same"` (within ± 1 percentage point) or `"worse"` than with all 30 features? Fill in the dictionary.

In [ ]:
pred = {
    "logistic regression": None,   # "better", "same" or "worse"
    "k-NN": None,
    "random forest": None,
}

### 🛠️ Task 5.1
Finish the loop: for every model and every `k` in `[None, 2, 10]` (`None` = no PCA) compute the mean cross-validated accuracy of `pipe` on `X, y` with `cv` and store it in `acc`.

<details><summary>💡 Hint</summary>

`acc = cross_val_score(pipe, X, y, cv=cv).mean()`. Use the whole `X, y` – cross-validation does the splitting.

</details>

In [ ]:
cv = StratifiedKFold(5, shuffle=True, random_state=RANDOM_STATE)
models = {
    "logistic regression": LogisticRegression(max_iter=5000),
    "k-NN": KNeighborsClassifier(5),
    "random forest": RandomForestClassifier(200, random_state=RANDOM_STATE),
}
results = {}

In [ ]:
for name, model in models.items():
    for k in [None, 2, 10]:
        steps = [StandardScaler()] + ([PCA(n_components=k)] if k else []) + [model]
        pipe = make_pipeline(*steps)
        # TODO: mean cross-validated accuracy of `pipe`
        acc = None
        results[(name, k)] = acc

In [ ]:
check(lambda: len(results) == 9 and all(v is not None for v in results.values()), "All 9 combinations computed.", "results should have 3 models x 3 settings = 9 entries.")

table = pd.DataFrame({name: {"no PCA": results[(name, None)], "PCA k=2": results[(name, 2)], "PCA k=10": results[(name, 10)]} for name in models}).T
display(table.round(3))

def verdict(d):
    return "better" if d > 0.01 else ("worse" if d < -0.01 else "same")

hits = 0
for name in models:
    actual = verdict(results[(name, 10)] - results[(name, None)])
    ok = pred.get(name) == actual
    hits += ok
    print(f"{name:20s} your guess: {str(pred.get(name)):7s} actual: {actual:7s} {'✓' if ok else '✗'}")
print("🔮 correct predictions:", hits, "/ 3")

**Question 4 🕵️.** (a) Does PCA **improve** the accuracy of any model? (b) What do we gain with PCA then, and what is the price of keeping only 2 components?

✍️ **Your answer:** *(double-click this cell and write here)*

## 6. Visualization: PCA, t-SNE, UMAP 

For the pictures we switch to the **digits** data (1797 images of 8×8 pixels = 64 features, 10 classes), where we can judge the clusters easily.

In [ ]:
digits = load_digits()
X_d, y_d = digits.data, digits.target

def plot2d(Z, title, xlabel, ylabel, ax=None):
    ax = ax or plt.subplots(figsize=(5.5, 5))[1]
    ax.scatter(Z[:, 0], Z[:, 1], c=y_d, cmap="tab10", s=6); ax.set_title(title); ax.grid(False)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)

def evaluate(Z):
    """(trustworthiness, 5-NN accuracy in 2D): does the 2D picture keep neighbours / classes?"""
    return trustworthiness(X_d, Z, n_neighbors=10), cross_val_score(KNeighborsClassifier(5), Z, y_d, cv=5).mean()

### 6.1 PCA picture (given)
Look at the axes: a PCA picture is labelled with the component and its share of variance.

In [ ]:
pca2 = PCA(n_components=2).fit(X_d)
Z_pca = pca2.transform(X_d)
ev = pca2.explained_variance_ratio_
plot2d(Z_pca, "PCA", f"PC1 ({ev[0]:.1%} of variance)", f"PC2 ({ev[1]:.1%})"); plt.show()

### 🛠️ Task 6.2 – t-SNE
`Z_tsne = TSNE(random_state=RANDOM_STATE).fit_transform(X_d)` (takes 10–30 s). Then compare PCA and t-SNE with `evaluate`.

In [ ]:
# TODO
Z_tsne = None

In [ ]:
check(lambda: Z_tsne.shape == (1797, 2), "t-SNE computed.", "Z_tsne should be (1797, 2).")
res = {"PCA": evaluate(Z_pca), "t-SNE": evaluate(Z_tsne)}
print(pd.DataFrame(res, index=["trustworthiness", "5-NN accuracy (2D)"]).T.round(3))
plot2d(Z_tsne, "t-SNE", "t-SNE 1 (no meaning)", "t-SNE 2 (no meaning)"); plt.show()

### 🛠️ Task 6.3 – UMAP
`Z_umap = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=RANDOM_STATE).fit_transform(X_d)`. (Needs `!pip install umap-learn` in Colab; the cell is skipped when UMAP is missing.)

In [ ]:
if HAS_UMAP:
    # TODO
    Z_umap = None

In [ ]:
if HAS_UMAP:
    res["UMAP"] = evaluate(Z_umap)
    plot2d(Z_umap, "UMAP", "UMAP 1 (no meaning)", "UMAP 2 (no meaning)"); plt.show()
else:
    print("UMAP skipped.")
print(pd.DataFrame(res, index=["trustworthiness", "5-NN accuracy (2D)"]).T.round(3))

**Question 5 🕵️.** (a) Why is the t-SNE picture so much "cleaner" than the PCA one? (b) Is t-SNE therefore the "better" method? (c) Why are the PCA axes labelled with the share of variance and the t-SNE axes "no meaning"?

✍️ **Your answer:** *(double-click this cell and write here)*

### 6.4 Perplexity (look, no code to write)
`perplexity` is roughly "how many neighbours each point considers". The same 1000 digits with three values:

In [ ]:
rng = np.random.RandomState(RANDOM_STATE)
sub = rng.choice(len(X_d), 1000, replace=False)
X_sub, y_sub = X_d[sub], y_d[sub]
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, p in zip(axes, [2, 30, 200]):
    Z = TSNE(perplexity=p, random_state=RANDOM_STATE).fit_transform(X_sub)
    ax.scatter(Z[:, 0], Z[:, 1], c=y_sub, cmap="tab10", s=6); ax.set_title(f"perplexity = {p}"); ax.grid(False)
plt.tight_layout(); plt.show()

Too small (2): scattered fragments, each digit falls apart. Reasonable (30): ten clear clusters. Too large (200): clusters merge and local detail is lost. **Always try several values and trust only the structure that stays stable.**

### 🕵️ 6.5 Does t-SNE hallucinate? (run and look)
A new data set: **500 points of pure random noise** in 20 dimensions. There is **no structure** at all. t-SNE with `perplexity` 2 and 30:

In [ ]:
X_pure_noise = np.random.RandomState(0).randn(500, 20)
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, p in zip(axes, [2, 30]):
    Z = TSNE(perplexity=p, random_state=RANDOM_STATE).fit_transform(X_pure_noise)
    ax.scatter(Z[:, 0], Z[:, 1], s=8); ax.set_title(f"pure noise, perplexity = {p}"); ax.grid(False)
plt.tight_layout(); plt.show()

**Question 6 🕵️.** There is nothing in these data. Do both pictures look "structureless"? What would happen if you showed the perplexity-2 picture to someone and said "we found small groups of similar customers"? How can you protect yourself?

✍️ **Your answer:** *(double-click this cell and write here)*

## 7. Latent space: encode, decode, denoise 

PCA with 10 components gives every digit a code of **10 numbers** – its position in a 10-dimensional **latent space**. We can go there (`transform` = encode) and back (`inverse_transform` = decode). What can we do with it?

### 🛠️ Task 7.1 – denoising
We damage the digits with strong random noise (`X_noisy`). Fit `PCA(n_components=10)` on the **clean** data `X_d` (`pca10`) and **encode and decode the noisy digits**: `X_den = decode(encode(X_noisy))`. The idea: noise has no structure, so it does not fit into the 10 main directions and is thrown away.

<details><summary>💡 Hint</summary>

`pca10 = PCA(n_components=10).fit(X_d)`; `X_den = pca10.inverse_transform(pca10.transform(X_noisy))`.

</details>

In [ ]:
X_noisy = X_d + np.random.RandomState(0).normal(0, 4, X_d.shape)

In [ ]:
# TODO
pca10 = None
X_den = None

In [ ]:
mse_noisy, mse_den = np.mean((X_noisy - X_d) ** 2), np.mean((X_den - X_d) ** 2)
check(lambda: mse_den < 0.6 * mse_noisy, f"The error to the clean images dropped from {mse_noisy:.1f} to {mse_den:.1f}.", "Encode (transform) and then decode (inverse_transform) the NOISY digits.")
fig, axes = plt.subplots(3, 8, figsize=(12, 5))
for row, (title, data) in enumerate([("clean", X_d), ("noisy", X_noisy), ("denoised", X_den)]):
    for j in range(8):
        axes[row, j].imshow(data[j].reshape(8, 8), cmap="gray_r"); axes[row, j].axis("off")
    axes[row, 0].set_title(title, loc="left", fontsize=11)
plt.tight_layout(); plt.show()

### 🛠️ Task 7.2 – walk through the latent space
Take one "3" and one "8". Encode both (`z_a`, `z_b`), create **8 points on the straight line** between them in the latent space (`z_path`, shape (8, 10)) and decode them into images `X_path`.

<details><summary>💡 Hint</summary>

`z_a, z_b = pca10.transform([a, b])`; `z_path = np.linspace(z_a, z_b, 8)`; `X_path = pca10.inverse_transform(z_path)`.

</details>

In [ ]:
a, b = X_d[y_d == 3][0], X_d[y_d == 8][0]

In [ ]:
# TODO
z_a, z_b = None, None
z_path = None
X_path = None

In [ ]:
check(lambda: z_path.shape == (8, 10) and X_path.shape == (8, 64), "8 points in the latent space decoded into 8 images.", "z_path should be (8, 10), X_path (8, 64).")
fig, axes = plt.subplots(1, 8, figsize=(12, 2))
for ax, img in zip(axes, X_path):
    ax.imshow(img.reshape(8, 8), cmap="gray_r"); ax.axis("off")
plt.show()

**Question 7 🕵️.** (a) Why does the denoising work, and when would it fail? (b) What do the images on the straight path between the 3 and the 8 look like? What does it say about the PCA latent space, and what would you want from a *better* latent space?

## 8. Would you use PCA? Three real situations (8 min)

No code now. For each situation decide **yes / no / test it**, and justify it in one or two sentences. Think about: the model, the number and type of features, interpretability, the amount of data. **Discuss with your neighbour** – you may disagree!

| | Situation |
|---|---|
| **A** | **Retail demand forecasting.** LightGBM predicts daily sales from 40 features: price, promo flag, weekday, holiday, sales lag 7 and 28 days, store ID, product category… The business wants to know *which factors drive sales*. |
| **B** | **Spectroscopy.** 150 samples, each with 2 000 neighbouring spectral channels; we predict a concentration with linear regression. |
| **C** | **Medical questionnaire.** 15 yes/no questions, logistic regression. The doctors want to read from the model how much each answer matters. |

✍️ **Your decisions** *(double-click and fill in: yes / no / test it + why)*

**A:** 

**B:** 

**C:** 

## 9. Wrap-up 

### Scoreboard 🔮
Run the cell to see how well you predicted.

In [ ]:
points = 0
if guess_k is not None and abs(guess_k - k95) <= 3:
    points += 1
points += hits
print(f"Your prediction score: {points} / 4  (1 for the number of components + 3 for the models)")
print("0–1: surprised by the data - good, that is how you learn.  2–3: solid intuition.  4: you could teach this!")

### Self-check
- [ ] I can describe the PCA workflow (split → scaler + PCA on train → choose k → model) and say why it lives in a pipeline.
- [ ] I can explain data leakage with an example, and I know what a pipeline protects me from.
- [ ] I can test whether PCA helps a given model and decide with numbers, not with a feeling.
- [ ] I know how to read (and how not to read) t-SNE/UMAP and why I need to try several parameter values.
- [ ] I can say for a new problem whether PCA is promising, and what I would compare it with.



